# Customer Churn & CLV Analytics Platform
**Python · Pandas · Scikit-learn · XGBoost · SHAP · Power BI**

Bu notebook, projede Power BI aşamasına kadar kullandığımız kodların sadeleştirilmiş tek-akış sürümüdür.

**Akış:** veri temizleme → feature engineering → RFM → churn etiketi → model karşılaştırma → threshold → SHAP → tüm müşterileri scoring → 90 günlük customer value → revenue at risk → aksiyon motoru → Power BI export.

> `expected_clv_90d`, probabilistic lifetime CLV değil; churn olasılığıyla düzeltilmiş 90 günlük beklenen müşteri değeridir.


### Reproducibility
Place `online_retail_II.xlsx` in `data/raw/` before running the notebook. The notebook exports Power BI-ready CSV files to `data/processed/`.


## 1. Imports ve ayarlar

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
PREDICTION_WINDOW_DAYS = 90

RAW_DATA_PATH = Path("../data/raw/online_retail_II.xlsx")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("SHAP:", shap.__version__)

## 2. Veriyi yükle ve kolonları standardize et

In [ ]:
df = pd.read_excel(RAW_DATA_PATH)

df.columns = (
    df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)

aliases = {
    "invoice": "invoice_no", "invoiceno": "invoice_no",
    "invoicedate": "invoice_date", "price": "unit_price",
    "unitprice": "unit_price", "customerid": "customer_id"
}
df = df.rename(columns={c: aliases.get(c, c) for c in df.columns})

required = [
    "invoice_no", "stockcode", "description", "quantity",
    "invoice_date", "unit_price", "customer_id", "country"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Eksik kolonlar: {missing}")

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df["customer_id"] = (
    df["customer_id"].astype("string")
      .str.replace(r"\.0$", "", regex=True).str.strip()
)
df["invoice_no"] = df["invoice_no"].astype("string").str.strip()
df["stockcode"] = df["stockcode"].astype("string").str.strip()

print("Rows:", f"{len(df):,}")
print("Customers:", f"{df['customer_id'].nunique(dropna=True):,}")
print("Date:", df["invoice_date"].min(), "→", df["invoice_date"].max())

## 3. Temizleme: geçerli satışlar ve iadeler

In [ ]:
df_clean = df.dropna(subset=["customer_id", "invoice_date"]).copy()

df_clean["is_cancellation"] = (
    df_clean["invoice_no"].str.upper().str.startswith("C", na=False)
    | (df_clean["quantity"] < 0)
)

sales_df = df_clean[
    (~df_clean["is_cancellation"])
    & (df_clean["quantity"] > 0)
    & (df_clean["unit_price"] > 0)
].copy()

returns_df = df_clean[df_clean["is_cancellation"]].copy()
sales_df["revenue"] = sales_df["quantity"] * sales_df["unit_price"]

print("Valid sales:", f"{len(sales_df):,}")
print("Returns/cancellations:", f"{len(returns_df):,}")
print("Sales customers:", f"{sales_df['customer_id'].nunique():,}")
print("Revenue:", f"£{sales_df['revenue'].sum():,.2f}")

## 4. Müşteri feature engineering

In [ ]:
def build_customer_features(sales, returns, snapshot_date):
    s = sales[sales["invoice_date"] <= snapshot_date].copy()
    r = returns[returns["invoice_date"] <= snapshot_date].copy()

    out = s.groupby("customer_id").agg(
        last_purchase=("invoice_date", "max"),
        first_purchase=("invoice_date", "min"),
        frequency=("invoice_no", "nunique"),
        monetary=("revenue", "sum"),
        total_items=("quantity", "sum"),
        unique_products=("stockcode", "nunique"),
    )

    out["recency"] = (snapshot_date - out["last_purchase"]).dt.days
    out["customer_tenure_days"] = (
        snapshot_date - out["first_purchase"]
    ).dt.days.clip(lower=0)
    out["avg_order_value"] = out["monetary"] / out["frequency"].clip(lower=1)

    rc = r.groupby("customer_id")["invoice_no"].nunique().rename("return_transactions")
    out = out.join(rc, how="left")
    out["return_transactions"] = out["return_transactions"].fillna(0)
    out["return_rate"] = (
        out["return_transactions"]
        / (out["frequency"] + out["return_transactions"]).clip(lower=1)
    )

    return out.reset_index().drop(columns=["last_purchase", "first_purchase"])

FEATURE_COLUMNS = [
    "recency", "frequency", "monetary", "total_items",
    "unique_products", "avg_order_value", "customer_tenure_days",
    "return_transactions", "return_rate"
]

## 5. RFM segmentasyonu

In [ ]:
snapshot_date = sales_df["invoice_date"].max()
rfm = build_customer_features(sales_df, returns_df, snapshot_date)[
    ["customer_id", "recency", "frequency", "monetary"]
].copy()

def qscore(series, higher_is_better=True):
    score = pd.qcut(
        series.rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]
    ).astype(int)
    return score if higher_is_better else 6 - score

rfm["r_score"] = qscore(rfm["recency"], False)
rfm["f_score"] = qscore(rfm["frequency"], True)
rfm["m_score"] = qscore(rfm["monetary"], True)

def rfm_segment(row):
    r, fm = row["r_score"], (row["f_score"] + row["m_score"]) / 2
    if r >= 4 and fm >= 4: return "Champions"
    if r >= 3 and fm >= 3: return "Loyal Customers"
    if r <= 2 and fm >= 4: return "Can't Lose Them"
    if r <= 2 and fm >= 2: return "At Risk"
    if r >= 4 and fm < 3: return "Potential Loyalists"
    if r == 3 and fm < 3: return "Need Attention"
    return "Lost Customers"

rfm["rfm_segment"] = rfm.apply(rfm_segment, axis=1)

rfm["rfm_segment"].value_counts()

## 6. Zamansal churn etiketi

In [ ]:
max_date = sales_df["invoice_date"].max()
cutoff_date = max_date - pd.Timedelta(days=PREDICTION_WINDOW_DAYS)

historical_sales = sales_df[sales_df["invoice_date"] <= cutoff_date]
historical_returns = returns_df[returns_df["invoice_date"] <= cutoff_date]
future_sales = sales_df[
    (sales_df["invoice_date"] > cutoff_date)
    & (sales_df["invoice_date"] <= max_date)
]

model_df = build_customer_features(
    historical_sales, historical_returns, cutoff_date
)

future_active = set(future_sales["customer_id"].unique())
model_df["churn"] = (~model_df["customer_id"].isin(future_active)).astype(int)

print("Max date:", max_date)
print("Cutoff:", cutoff_date)
print("Model rows:", len(model_df))
print(model_df["churn"].value_counts())
print("Churn rate:", f"{model_df['churn'].mean():.2%}")

## 7. Train / validation / test

In [ ]:
X = model_df[FEATURE_COLUMNS].copy()
y = model_df["churn"].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20,
    stratify=y_trainval, random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 8. Model karşılaştırması

In [ ]:
def evaluate(name, model, X_eval, y_eval, threshold=0.50):
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= threshold).astype(int)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred, zero_division=0),
        "F1": f1_score(y_eval, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_eval, prob),
        "PR_AUC": average_precision_score(y_eval, prob)
    }

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])
rf = RandomForestClassifier(
    n_estimators=400, min_samples_leaf=3,
    class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
)
xgb = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    eval_metric="logloss", random_state=RANDOM_STATE
)

models = {"Logistic Regression": lr, "Random Forest": rf, "XGBoost": xgb}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    results.append(evaluate(name, model, X_val, y_val))

pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)

## 9. Random Forest threshold optimizasyonu

In [ ]:
val_prob = rf.predict_proba(X_val)[:, 1]
rows = []

for t in np.arange(0.20, 0.81, 0.05):
    pred = (val_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(rows)
best_threshold = float(
    threshold_df.loc[threshold_df["f1"].idxmax(), "threshold"]
)

display(threshold_df)
print("Selected threshold:", round(best_threshold, 2))

## 10. Final Random Forest ve test sonucu

In [ ]:
rf_final = RandomForestClassifier(
    n_estimators=400, min_samples_leaf=3,
    class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
)
rf_final.fit(X_trainval, y_trainval)

final_results = evaluate(
    "Random Forest", rf_final, X_test, y_test, threshold=best_threshold
)
final_results

## 11. SHAP

In [ ]:
explainer = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_values_churn = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values_churn = shap_values[:, :, 1]
else:
    shap_values_churn = shap_values

shap_importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "mean_abs_shap": np.abs(shap_values_churn).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance)

shap.summary_plot(shap_values_churn, X_test)

## 12. Tüm müşterileri scoring

In [ ]:
customer_360_all = build_customer_features(
    sales_df, returns_df, snapshot_date
)

customer_360_all = customer_360_all.merge(
    rfm[["customer_id", "rfm_segment"]],
    on="customer_id", how="left"
)

customer_360_all["value_segment"] = pd.qcut(
    customer_360_all["monetary"].rank(method="first"),
    q=4,
    labels=["Low Value", "Medium Value", "High Value", "VIP Value"]
).astype(str)

X_all = customer_360_all[FEATURE_COLUMNS]
customer_360_all["churn_probability"] = rf_final.predict_proba(X_all)[:, 1]
customer_360_all["predicted_churn"] = (
    customer_360_all["churn_probability"] >= best_threshold
).astype(int)

def risk_level(p):
    if p >= 0.80: return "Critical Risk"
    if p >= 0.60: return "High Risk"
    if p >= 0.40: return "Medium Risk"
    return "Low Risk"

customer_360_all["risk_level"] = (
    customer_360_all["churn_probability"].apply(risk_level)
)

print("Customers scored:", len(customer_360_all))

## 13. 90 günlük customer value ve Revenue at Risk

In [ ]:
customer_360_all["purchase_rate_per_day"] = (
    customer_360_all["frequency"]
    / customer_360_all["customer_tenure_days"].clip(lower=1)
)
customer_360_all["expected_orders_90d"] = (
    customer_360_all["purchase_rate_per_day"] * 90
)
customer_360_all["expected_revenue_90d"] = (
    customer_360_all["expected_orders_90d"]
    * customer_360_all["avg_order_value"]
)
customer_360_all["expected_clv_90d"] = (
    customer_360_all["expected_revenue_90d"]
    * (1 - customer_360_all["churn_probability"])
)
customer_360_all["revenue_at_risk_90d"] = (
    customer_360_all["expected_revenue_90d"]
    * customer_360_all["churn_probability"]
)
customer_360_all["retention_priority_score"] = (
    customer_360_all["revenue_at_risk_90d"]
)

## 14. Retention Action Engine

In [ ]:
def retention_action(row):
    risk = row["churn_probability"]
    value = row["expected_clv_90d"]

    if risk >= 0.80 and value >= 5000:
        return "VIP Human Retention"
    if risk >= 0.60 and value >= 2000:
        return "Personalized Retention Offer"
    if risk >= 0.60:
        return "Automated Win-Back Campaign"
    if risk < 0.40 and value >= 5000:
        return "VIP Loyalty & Upsell"
    if risk < 0.40:
        return "Standard Engagement"
    return "Monitor"

customer_360_all["recommended_action"] = (
    customer_360_all.apply(retention_action, axis=1)
)

customer_360_all["recommended_action"].value_counts()

## 15. Power BI final dataset

In [ ]:
FINAL_COLUMNS = [
    "customer_id", "recency", "frequency", "monetary",
    "total_items", "unique_products", "avg_order_value",
    "customer_tenure_days", "return_transactions", "return_rate",
    "churn_probability", "predicted_churn", "risk_level",
    "rfm_segment", "value_segment",
    "expected_orders_90d", "expected_revenue_90d",
    "expected_clv_90d", "revenue_at_risk_90d",
    "retention_priority_score", "recommended_action"
]

missing = [c for c in FINAL_COLUMNS if c not in customer_360_all.columns]
assert not missing, f"Eksik kolonlar: {missing}"

customer_360_all = customer_360_all[FINAL_COLUMNS].copy()

print("Rows:", len(customer_360_all))
print("Unique customers:", customer_360_all["customer_id"].nunique())
print("Duplicate IDs:", customer_360_all["customer_id"].duplicated().sum())
print("Missing values:", customer_360_all.isna().sum().sum())

## 16. Final özet

In [ ]:
print("=" * 55)
print("FINAL CUSTOMER 360 - ALL CUSTOMERS")
print("=" * 55)
print(f"Customers: {len(customer_360_all):,}")
print(f"Predicted churn customers: {customer_360_all['predicted_churn'].sum():,}")
print(f"Predicted churn rate: {customer_360_all['predicted_churn'].mean():.2%}")
print(f"Average churn probability: {customer_360_all['churn_probability'].mean():.2%}")
print(f"Expected 90-Day CLV: £{customer_360_all['expected_clv_90d'].sum():,.2f}")
print(f"90-Day Revenue at Risk: £{customer_360_all['revenue_at_risk_90d'].sum():,.2f}")

print("\nRisk distribution:")
print(customer_360_all["risk_level"].value_counts())

print("\nRecommended actions:")
print(customer_360_all["recommended_action"].value_counts())

## 17. Top 20 retention targets

In [ ]:
top_20_retention_all = (
    customer_360_all
    .sort_values("revenue_at_risk_90d", ascending=False)
    .head(20)
)

top_20_retention_all[[
    "customer_id", "churn_probability", "risk_level", "monetary",
    "value_segment", "expected_clv_90d", "revenue_at_risk_90d",
    "recommended_action"
]]

## 18. Power BI export

In [ ]:
risk_summary_all = (
    customer_360_all.groupby("risk_level", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        revenue_at_risk=("revenue_at_risk_90d", "sum"),
        expected_clv=("expected_clv_90d", "sum"),
        avg_churn_probability=("churn_probability", "mean")
    )
)

action_summary_all = (
    customer_360_all.groupby("recommended_action", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        revenue_at_risk=("revenue_at_risk_90d", "sum"),
        expected_clv=("expected_clv_90d", "sum"),
        avg_churn_probability=("churn_probability", "mean")
    )
)

customer_360_all.to_csv(PROCESSED_DIR / "customer_360_all.csv", index=False)
top_20_retention_all.to_csv(PROCESSED_DIR / "top_20_retention_all.csv", index=False)
risk_summary_all.to_csv(PROCESSED_DIR / "risk_summary_all.csv", index=False)
action_summary_all.to_csv(PROCESSED_DIR / "action_summary_all.csv", index=False)
shap_importance.to_csv(PROCESSED_DIR / "shap_importance.csv", index=False)

for f in PROCESSED_DIR.glob("*.csv"):
    print("✓", f.name)

## Power BI notu
`customer_360_all.csv` ana dashboard tablosudur.

- `churn_probability`, `return_rate` → Decimal Number, yüzde biçimi
- `monetary`, `avg_order_value`, `expected_revenue_90d`, `expected_clv_90d`, `revenue_at_risk_90d` → Decimal/Currency
- `risk_level`, `rfm_segment`, `value_segment`, `recommended_action` → Text
- `predicted_churn` → Whole Number

Test-seti model metrikleri ile tüm müşteri scoring sonuçlarını raporda ayrı tutun.
